## Gold Consolidado — Data Quality (todas as tabelas)

In [0]:
%run ../../utils/utils

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from functools import reduce
 
print("Iniciando processamento da Camada Gold Consolidada - Data Quality")

## Lista de tabelas e mapeamento de ID por tabela

In [0]:
TABELAS = [
    "ecommerce_categorias",
    "ecommerce_clientes",
    "ecommerce_enderecos",
    "ecommerce_itens_pedido",
    "ecommerce_pedidos",
    "ecommerce_produtos",
    "ecommerce_rastreamento",
]

IDS_POR_TABELA = {
    "ecommerce_categorias": "id_categoria",
    "ecommerce_clientes": "id_cliente",
    "ecommerce_enderecos": "id_endereco",
    "ecommerce_itens_pedido": "id_item_pedido",
    "ecommerce_pedidos": "id_pedido",
    "ecommerce_produtos": "sku",
    "ecommerce_rastreamento": "id_rastreamento",
}
 
print(f"Processando Gold para {len(TABELAS)} tabelas: {TABELAS}")

## Regras por tabela

In [0]:
lista_df_regras = []
lista_df_saude_hora = []
lista_df_saude_percentual = []
lista_df_quarentena = []
 
for TABELA_ALVO in TABELAS:
    print(f"\n===== Processando: {TABELA_ALVO} =====")
 
    # --------------------------------------------------------------------
    # 1. RESUMO POR REGRA (a partir de dq_monitoring_logs)
    # --------------------------------------------------------------------
    if delta_existe(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS):
        df_logs = ler_delta(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS) \
            .filter(F.col("tabela") == TABELA_ALVO)
 
        if df_logs.count() > 0:
            df_gold_regras = df_logs \
                .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
                .groupBy("data_execucao", "tabela", "regra", "severidade") \
                .agg(
                    # qtd_totalfalhas = quantidade REAL de registros que falharam nessa regra (o número correto de erros)
                    F.sum("qtd_totalfalhas").alias("total_falhas"),
                    # qtd_registros_falhos agora é uma FLAG (1 por execução em que a regra falhou) — soma = nº de execuções com falha
                    F.sum("qtd_registros_falhos").alias("qtd_execucoes_com_falha"),
                    F.sum("qtd_registros_total").alias("total_processado")
                ) \
                .withColumn("perc_falha", F.round((F.col("total_falhas") / F.col("total_processado")) * 100, 2))
 
            # Tabela por entidade removida: mantemos apenas a consolidada (gold_dq_regras_consolidado),
            # filtrável no Looker via WHERE tabela = 'nome_desejado'.
            lista_df_regras.append(df_gold_regras)
        else:
            print(f"  -> Aviso: nenhum log de falha para {TABELA_ALVO}.")
    else:
        print("  -> Aviso: dq_monitoring_logs não encontrada.")
 
    # --------------------------------------------------------------------
    # 2. SAÚDE POR HORA (volume de registros limpos na Silver)
    # --------------------------------------------------------------------
    if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
        df_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
 
        df_gold_saude = df_silver \
            .withColumn("data_hora", F.date_trunc("hour", F.col("silver_processed_at"))) \
            .groupBy("data_hora") \
            .agg(F.count("*").alias("qtd_limpos")) \
            .withColumn("tabela", F.lit(TABELA_ALVO)) \
            .withColumn("qtd_total", F.col("qtd_limpos")) \
            .withColumn("perc_limpos", F.lit(100.0)) \
            .select("data_hora", "tabela", "qtd_total", "qtd_limpos", "perc_limpos")
 
        # Tabela por entidade removida: mantemos apenas a consolidada (gold_dq_saude_consolidado),
        # filtrável no Looker via WHERE tabela = 'nome_desejado'.
        lista_df_saude_hora.append(df_gold_saude)
    else:
        print(f"  -> Aviso: Silver de {TABELA_ALVO} não encontrada.")
 
    # --------------------------------------------------------------------
    # 3. SAÚDE PERCENTUAL (% da Bronze que virou Silver)
    # --------------------------------------------------------------------
    id_col = IDS_POR_TABELA.get(TABELA_ALVO)
    if id_col is None:
        print(f"  -> Aviso: coluna de ID não mapeada para {TABELA_ALVO}. Saúde percentual ignorada.")
        continue
 
    qtd_bronze_distintos, qtd_quarentena_saude = 0, 0
 
    set_ids_bronze_atual = set()
    if delta_existe("bronze", TABELA_ALVO, STORAGE_OPTIONS):
        pdf_ids_bronze = (
            ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
            .select(id_col).where(F.col(id_col).isNotNull()).distinct()
            .toPandas()
        )
        set_ids_bronze_atual = set(pdf_ids_bronze[id_col])
        qtd_bronze_distintos = len(set_ids_bronze_atual)
 
    set_ids_silver_atual = set()
    if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
        pdf_ids_silver = (
            ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
            .select(id_col).where(F.col(id_col).isNotNull()).distinct()
            .toPandas()
        )
        set_ids_silver_atual = set(pdf_ids_silver[id_col])
 
    if set_ids_bronze_atual and delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        pdf_quarentena_ids = (
            ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            .select(id_col).where(F.col(id_col).isNotNull()).distinct()
            .toPandas()
        )
        set_quarentena_ids = set(pdf_quarentena_ids[id_col])
        qtd_quarentena_saude = len((set_ids_bronze_atual & set_quarentena_ids) - set_ids_silver_atual)
 
    if qtd_bronze_distintos > 0:
        qtd_quarentena_saude = min(qtd_quarentena_saude, qtd_bronze_distintos)
        percentual_invalidado = round((qtd_quarentena_saude / qtd_bronze_distintos) * 100, 2)
        percentual_saude = round(100 - percentual_invalidado, 2)
 
        df_saude_pct = spark.createDataFrame(
            [(TABELA_ALVO, qtd_bronze_distintos, qtd_quarentena_saude, percentual_invalidado, percentual_saude)],
            ["tabela", "total_bronze_distintos", "total_quarentena", "percentual_invalidado", "percentual_saude"]
        ).withColumn("data_verificacao", F.current_timestamp())
 
        # Tabela por entidade removida: mantemos apenas a consolidada (gold_saude_percentual_consolidado),
        # filtrável no Looker via WHERE tabela = 'nome_desejado'.
        lista_df_saude_percentual.append(df_saude_pct)
        print(f"  -> Saúde: {percentual_saude}% passou para a Silver, {percentual_invalidado}% invalidado "
              f"({qtd_quarentena_saude} de {qtd_bronze_distintos}).")
    else:
        print(f"  -> Aviso: Bronze de {TABELA_ALVO} vazia, saúde percentual não calculada.")
 
    # --------------------------------------------------------------------
    # 4. QUARENTENA (exporta os registros reprovados para a Gold/SQL Server)
    # --------------------------------------------------------------------
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena_silver = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        qtd_rejeitados = df_quarentena_silver.count()
 
        if qtd_rejeitados > 0:
            # Trata colunas NullType (schema totalmente vazio) para compatibilidade com o SQL Server
            colunas_corrigidas = [
                F.col(campo.name).cast("string") if str(campo.dataType).lower().startswith("null") else F.col(campo.name)
                for campo in df_quarentena_silver.schema.fields
            ]
            df_quarentena_pronta = df_quarentena_silver.select(*colunas_corrigidas)

            # Adiciona a coluna "tabela" para permitir filtrar no Looker (WHERE tabela = 'nome_desejado')
            # depois de consolidar todas as entidades em uma única tabela Gold.
            if "tabela" not in df_quarentena_pronta.columns:
                df_quarentena_pronta = df_quarentena_pronta.withColumn("tabela", F.lit(TABELA_ALVO))

            lista_df_quarentena.append(df_quarentena_pronta)
            print(f"  -> Quarentena: {qtd_rejeitados} registros de {TABELA_ALVO} preparados para consolidação.")
        else:
            print(f"  -> Quarentena de {TABELA_ALVO} vazia. Nada a gravar.")
    else:
        print(f"  -> Quarentena de {TABELA_ALVO} ainda não existe.")


## Consolidar (unionByName) e gravar as 3 tabelas Gold únicas

In [0]:
if lista_df_regras:
    df_regras_consolidado = reduce(lambda a, b: a.unionByName(b), lista_df_regras)
    gravar_gold_completo(df_regras_consolidado, "gold_dq_regras_consolidado")
 
if lista_df_saude_hora:
    df_saude_hora_consolidado = reduce(lambda a, b: a.unionByName(b), lista_df_saude_hora)
    gravar_gold_completo(df_saude_hora_consolidado, "gold_dq_saude_consolidado")
 
if lista_df_saude_percentual:
    df_saude_pct_consolidado = reduce(lambda a, b: a.unionByName(b), lista_df_saude_percentual)
    gravar_gold_completo(df_saude_pct_consolidado, "gold_saude_percentual_consolidado")
    display(df_saude_pct_consolidado.orderBy("percentual_invalidado", ascending=False))

if lista_df_quarentena:
    # allowMissingColumns=True: cada entidade tem colunas de negócio diferentes
    # (ex: clientes tem "email", produtos tem "sku"); colunas ausentes em uma
    # entidade viram NULL nas linhas das demais, em vez de dar erro de schema.
    df_quarentena_consolidado = reduce(
        lambda a, b: a.unionByName(b, allowMissingColumns=True), lista_df_quarentena
    )
    gravar_gold_completo(df_quarentena_consolidado, "gold_dq_quarentena_consolidado")

print("\n===== GOLD FINALIZADO — APENAS 4 TABELAS CONSOLIDADAS "
      "(gold_dq_regras_consolidado, gold_dq_saude_consolidado, "
      "gold_saude_percentual_consolidado, gold_dq_quarentena_consolidado) =====")